In [ ]:
import pandas as pd
import joblib
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
df_old = pd.read_csv('../../../../data/preprocessed/selected_features_dataset_V3.csv')

In [ ]:
df_old.columns.to_list()

In [ ]:
df = pd.read_csv('../../../../data/processed/100k.csv')
X = df.drop(['price_per_m2'], axis=1)
y = df['price_per_m2']

In [ ]:
LABEL_ENCODERS_PATH = "../../../../data/processed/label_encoders_v3.pkl"
SELECTED_FEATURES = ['address_line_2',
 'h_id',
 'n_cafe_5km',
 'near_Royal_Palace_in_km',
 'n_gas_station_5km',
 'near_Phsar_Tmey_in_km',
 'near_Vattanac_Tower_in_km',
 'near_Phsar_kandal_in_km',
 'near_Phsar_Chas_in_km',
 'near_Wat_Phnom_in_km',
 'near_Sisowath_Riverside_Park_in_km',
 'near_Bassac_Lane_in_km',
 'near_Olympic_Stadium_in_km',
 'near_Boeng_Keng_Kang_1_in_km',
 'n_secondary_school_5km']

In [ ]:
label_encoders = joblib.load(LABEL_ENCODERS_PATH)


In [ ]:
print(label_encoders)

In [ ]:
features = X.copy()

In [ ]:
encoded_values = {}

for col, encoder in label_encoders.items():
    if col in features:
        def encode_value(val):
            if val in encoder.classes_:
                return encoder.transform([val])[0]
            else:
                return -1  # Handle unseen label
        
        # Apply encoding row-wise
        features[col] = features[col].apply(encode_value)
        encoded_values[col] = features[col]


In [ ]:
X.head()

In [ ]:
features.head()

In [ ]:
features = features[SELECTED_FEATURES]

In [ ]:
def predict_with_models(X_processed, models):
    predictions = {}
    for name, model in models.items():
        predictions[name] = model.predict(X_processed)
    return pd.DataFrame(predictions)

In [ ]:
lr_model = joblib.load("../../../../models/linear_regression/linear_regression_model_v4.joblib")
rf_model = joblib.load("../../../../models/random_forest/random_forest_model_v7_label.joblib")
xgb_model = joblib.load("../../../../models/xgboost/xgboost_model_label_v7.joblib")
svr_model = joblib.load("../../../../models/svr/svr_model_label_v5.joblib")
knn_model = joblib.load("../../../../models/knn/knn_model_v2.joblib")
br_model = joblib.load("../../../../models/bayesian_ridge/bayesian_ridge_model_label_v2.joblib")

In [ ]:
models = {
    "LR": lr_model,
    "RF": rf_model,
    "XGBoost": xgb_model,
    "SVR": svr_model,
    "KNN": knn_model,
    "BR": br_model
}

In [ ]:
def get_used_feature_names(models_dict):
    """Get the feature names used by each model"""
    feature_info = {}
    
    for name, model in models_dict.items():
        try:
            # Scikit-learn models (v1.0+)
            if hasattr(model, 'feature_names_in_'):
                feature_info[name] = list(model.feature_names_in_)
            
            # XGBoost models
            elif hasattr(model, 'get_booster'):
                feature_info[name] = model.get_booster().feature_names
            
            # Models with coefficients (Linear, BayesianRidge)
            elif hasattr(model, 'coef_'):
                if hasattr(model, 'feature_names_in_'):
                    feature_info[name] = list(model.feature_names_in_)
                else:
                    feature_info[name] = [f"feature_{i}" for i in range(len(model.coef_))]
            
            # Models with feature_importances_
            elif hasattr(model, 'feature_importances_'):
                if hasattr(model, 'feature_names_in_'):
                    feature_info[name] = list(model.feature_names_in_)
                else:
                    feature_info[name] = [f"feature_{i}" for i in range(len(model.feature_importances_))]
            
            else:
                feature_info[name] = "Feature names not available"
                
        except Exception as e:
            feature_info[name] = f"Error: {str(e)}"
    
    return pd.DataFrame.from_dict(feature_info, orient='index').T

# Usage after model training:
feature_names_df = get_used_feature_names(models)
print("Features used by each model:")
display(feature_names_df)

In [ ]:
preds_df = predict_with_models(features, models)

In [ ]:
preds_df

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero
    return np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-8, None))) * 100

def evaluate_models(y_true, predictions_df):
    results = []

    for model_name in predictions_df.columns:
        y_pred = predictions_df[model_name]
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred)

        results.append({
            'Model': model_name,
            'MAE': mae,
            'MSE': mse,
            'RMSE': rmse,
            'R2': r2,
            'MAPE (%)': mape
        })

    return pd.DataFrame(results).sort_values(by='RMSE')


In [ ]:
evaluation_results = evaluate_models(y, preds_df)

In [ ]:
print(evaluation_results)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Your evaluation results DataFrame
# Make sure `evaluation_results` is already defined and contains columns: Model, MAE, MSE, RMSE, R2, MAPE (%)
metrics_to_plot = ['MAE', 'RMSE', 'MAPE (%)']

# Convert wide to long format for plotting
df_long = evaluation_results.melt(id_vars='Model', value_vars=metrics_to_plot,
                                  var_name='Metric', value_name='Value')

# Sort models for consistent display
models = df_long['Model'].unique()
metrics = df_long['Metric'].unique()

# Define bar width and positions
bar_width = 0.2
x = range(len(models))

# Plotting
plt.figure(figsize=(12, 6))
custom_colors = {
    'MAE': "#1D70B3",       # Blue
    'RMSE': "#EDBE31",      # Green
    'MAPE (%)': "#918E8E"   # Purple
}

for i, metric in enumerate(metrics):
    data = df_long[df_long['Metric'] == metric]
    x_positions = [pos + i * bar_width for pos in x]
    bars = plt.bar(x_positions, data['Value'], width=bar_width, label=metric, color=custom_colors[metric])

    # Add number labels on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height,
                 f'{height:.2f}', ha='center', va='bottom', fontsize=8)

# Configure x-axis
plt.xticks([pos + bar_width * (len(metrics) - 1) / 2 for pos in x], models)
plt.xlabel('Model')
plt.ylabel('Metric Value')
plt.title('Model Performance Metrics')
plt.legend(title='Metric')
plt.tight_layout()
plt.show()


In [ ]:
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-8, None))) * 100

def evaluate_by_location(X_raw, y_true, predictions_df, location_col='address_locality'):
    results = []

    for model_name in predictions_df.columns:
        df = X_raw.copy()
        df['y_true'] = y_true
        df['y_pred'] = predictions_df[model_name]
        df['abs_error'] = np.abs(df['y_true'] - df['y_pred'])
        df['squared_error'] = (df['y_true'] - df['y_pred']) ** 2
        df['abs_pct_error'] = np.abs((df['y_true'] - df['y_pred']) / np.clip(df['y_true'], 1e-8, None)) * 100

        grouped = df.groupby(location_col).agg(
            count=('y_true', 'size'),
            MAE=('abs_error', 'mean'),
            MSE=('squared_error', 'mean'),
            RMSE=('squared_error', lambda x: np.sqrt(np.mean(x))),
            MAPE=('abs_pct_error', 'mean')
        ).reset_index()

        # Calculate R2 for each group, safely skipping groups with NaNs or no variation in y_true
        r2_values = []
        for loc in grouped[location_col]:
            group = df[df[location_col] == loc][['y_true', 'y_pred']].dropna()
            if not group.empty and group['y_true'].nunique() > 1:
                r2 = r2_score(group['y_true'], group['y_pred'])
            else:
                r2 = np.nan
            r2_values.append(r2)

        grouped['R2'] = r2_values
        grouped['Model'] = model_name
        results.append(grouped)

    return pd.concat(results, ignore_index=True)

In [ ]:
grouped_by_location_df = evaluate_by_location(X, y, preds_df)

knn_results = grouped_by_location_df[grouped_by_location_df['Model'] == 'KNN']

In [ ]:
knn_results

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_old = df_old.drop(['price_per_m2'], axis=1)
y_old = df_old['price_per_m2']
X_train_old, X_test_old, y_train_old, y_test_old = train_test_split(X_old, y_old, test_size=0.2, random_state=42)

In [ ]:
X_test_old

In [ ]:
models = {
    "LR": lr_model,
    "RF": rf_model,
    "XGBoost": xgb_model,
    "SVR": svr_model,
    "KNN": knn_model,
    "BR": br_model
}

In [ ]:
preds_df_old = predict_with_models(X_test_old, models)

In [ ]:
evaluation_results_old = evaluate_models(y_test_old, preds_df_old)

In [ ]:
evaluation_results_old

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Select metrics
metrics_to_plot = ['MAE', 'RMSE', 'MAPE (%)']

# Sort evaluation_results_old by MAE
sorted_df = evaluation_results_old.sort_values(by='MAE', ascending=True)
sorted_models = sorted_df['Model'].tolist()

# Melt for long format plotting
df_long = sorted_df.melt(id_vars='Model', value_vars=metrics_to_plot,
                         var_name='Metric', value_name='Value')

# Ensure model order is preserved
df_long['Model'] = pd.Categorical(df_long['Model'], categories=sorted_models, ordered=True)

# Bar setup
bar_width = 0.2
models = df_long['Model'].cat.categories.tolist()
metrics = df_long['Metric'].unique()
x = list(range(len(models)))

# Colors
custom_colors = {
    'MAE': "#1D70B3",       # Blue
    'RMSE': "#EDBE31",      # Yellow-Gold
    'MAPE (%)': "#918E8E"   # Gray
}

# Plot
plt.figure(figsize=(12, 6))
for i, metric in enumerate(metrics):
    data = df_long[df_long['Metric'] == metric]
    x_positions = [pos + i * bar_width for pos in x]
    bars = plt.bar(x_positions, data['Value'], width=bar_width,
                   label=metric, color=custom_colors[metric])

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height,
                 f'{height:.2f}', ha='center', va='bottom', fontsize=8)

# X-axis and labels
plt.xticks([pos + bar_width * (len(metrics) - 1) / 2 for pos in x], models)
plt.xlabel('Model')
plt.ylabel('Metric Value')
plt.title('Model Performance Metrics Bar')
plt.legend(title='Metric')
plt.tight_layout()
plt.show()


In [ ]:
y_test_old.isnull().sum()

In [ ]:
grouped_by_location_df_old = evaluate_by_location(X_test_old, y_test_old, preds_df_old, location_col="address_line_2")

knn_results = grouped_by_location_df_old[grouped_by_location_df_old['Model'] == 'RF']

In [ ]:
knn_results

In [ ]:
encoder = label_encoders['address_line_2']
knn_results['address_line_2'] = encoder.inverse_transform(knn_results['address_line_2'].astype(int))

In [ ]:
total_count = knn_results['count'].sum()
print("Total count:", total_count)


In [ ]:
knn_results

In [ ]:
knn_evaluation_results_old = evaluation_results_old[evaluation_results_old['Model'] == 'KNN']
knn_evaluation_results_old

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-8, None))) * 100

def evaluate_by_price_range(X_raw, y_true, predictions_df, price_col='y_true'):
    results = []

    # Define price bins (you can adjust bins for better accuracy)
    price_bins = [40, 100, 200, 500, 1000, 2000, 4000, 6000, 8000, 10000, 12000]
    price_labels = ['40-100', '100-200', '200-500', '500-1k', '1k-2k', '2k-4k', '4k-6k', '6k-8k', '8k-10k', '10k-12k']

    for model_name in predictions_df.columns:
        df = X_raw.copy()
        df['y_true'] = y_true
        df['y_pred'] = predictions_df[model_name]
        df['abs_error'] = np.abs(df['y_true'] - df['y_pred'])
        df['squared_error'] = (df['y_true'] - df['y_pred']) ** 2
        df['abs_pct_error'] = np.abs((df['y_true'] - df['y_pred']) / np.clip(df['y_true'], 1e-8, None)) * 100

        # Assign price ranges to y_true
        df['price_range'] = pd.cut(df['y_true'], bins=price_bins, labels=price_labels, include_lowest=True)

        grouped = df.groupby('price_range').agg(
            count=('y_true', 'size'),
            MAE=('abs_error', 'mean'),
            MSE=('squared_error', 'mean'),
            RMSE=('squared_error', lambda x: np.sqrt(np.mean(x))),
            MAPE=('abs_pct_error', 'mean')
        ).reset_index()

        # Calculate R2 for each price range, handling NaNs and constant values
        r2_values = []
        for pr in grouped['price_range']:
            group = df[df['price_range'] == pr][['y_true', 'y_pred']].dropna()
            if not group.empty and group['y_true'].nunique() > 1:
                r2 = r2_score(group['y_true'], group['y_pred'])
            else:
                r2 = np.nan
            r2_values.append(r2)

        grouped['R2'] = r2_values
        grouped['Model'] = model_name
        results.append(grouped)

    return pd.concat(results, ignore_index=True)


In [ ]:
grouped_by_price_df = evaluate_by_price_range(X_test_old, y_test_old, preds_df)
knn_price_results = grouped_by_price_df[grouped_by_price_df['Model'] == 'RF']

In [ ]:
knn_price_results

In [ ]:
def evaluate_by_price_quantiles(X_raw, y_true, predictions_df, n_bins=10):
    results = []
    quantile_bins = pd.qcut(y_true, q=n_bins, duplicates='drop')
    
    for model_name in predictions_df.columns:
        df = X_raw.copy()
        df['y_true'] = y_true
        df['y_pred'] = predictions_df[model_name]
        df['abs_error'] = np.abs(df['y_true'] - df['y_pred'])
        df['squared_error'] = (df['y_true'] - df['y_pred']) ** 2
        df['abs_pct_error'] = np.abs((df['y_true'] - df['y_pred']) / np.clip(df['y_true'], 1e-8, None)) * 100
        
        df['price_range'] = quantile_bins

        grouped = df.groupby('price_range').agg(
            count=('y_true', 'size'),
            MAE=('abs_error', 'mean'),
            MSE=('squared_error', 'mean'),
            RMSE=('squared_error', lambda x: np.sqrt(np.mean(x))),
            MAPE=('abs_pct_error', 'mean')
        ).reset_index()

        r2_values = []
        for pr in grouped['price_range']:
            group = df[df['price_range'] == pr][['y_true', 'y_pred']].dropna()
            if not group.empty and group['y_true'].nunique() > 1:
                r2 = r2_score(group['y_true'], group['y_pred'])
            else:
                r2 = np.nan
            r2_values.append(r2)

        grouped['R2'] = r2_values
        grouped['Model'] = model_name
        results.append(grouped)

    return pd.concat(results, ignore_index=True)


In [ ]:
grouped_by_bin_df = evaluate_by_price_quantiles(X_test_old, y_test_old, preds_df)
knn_bin_results = grouped_by_bin_df[grouped_by_bin_df['Model'] == 'KNN']
knn_bin_results

In [ ]:
grouped_by_bin_df = evaluate_by_price_quantiles(X, y, preds_df)
knn_results_by_bin = grouped_by_bin_df[grouped_by_bin_df['Model'] == 'KNN']
knn_results_by_bin

# GROUP DATA

In [ ]:
grouped_by_price_df = evaluate_by_price_range(X, y, preds_df)
knn_results_by_price = grouped_by_price_df[grouped_by_price_df['Model'] == 'RF']
knn_results_by_price.loc[:, knn_results_by_price.columns != 'count']

In [ ]:
grouped_by_price_df = evaluate_by_location(X, y, preds_df, location_col="address_line_2")
knn_results_by_price = grouped_by_price_df[grouped_by_price_df['Model'] == 'KNN']
pd.options.display.float_format = '{:.2f}'.format
knn_results_by_price.loc[:, knn_results_by_price.columns != 'count']

In [ ]:
print(knn_results_by_price.loc[:, knn_results_by_price.columns != 'count'])


In [ ]:
df = knn_results_by_price.loc[:, knn_results_by_price.columns != 'count'].copy()

# Round all numeric columns to 2 decimals
num_cols = df.select_dtypes(include=['float', 'int']).columns
df[num_cols] = df[num_cols].round(2)

# Sort by MAE ascending
df_sorted = df.sort_values(by='MAE', ascending=True)

print(df_sorted)

In [ ]:
grouped_by_price_df = evaluate_by_price_quantiles(X, y, preds_df)
knn_results_by_price = grouped_by_price_df[grouped_by_price_df['Model'] == 'KNN']
knn_results_by_price.loc[:, knn_results_by_price.columns != 'count']

In [ ]:
df = knn_results_by_price.loc[:, knn_results_by_price.columns != 'count'].copy()

# Round all numeric columns to 2 decimals
num_cols = df.select_dtypes(include=['float', 'int']).columns
df[num_cols] = df[num_cols].round(2)

# Sort by MAE ascending
df_sorted = df.sort_values(by='MAE', ascending=True)

print(df_sorted)

In [ ]:
def format_interval_as_int(interval):
    left = int(interval.left)
    right = int(interval.right)
    return f"({left}, {right}]"

df_sorted['price_range'] = df_sorted['price_range'].apply(format_interval_as_int)

print(df_sorted)
